In [ ]:
!pip install "hoct[bioio]" -q
!pip install numpy==2.0.2 -q

In [ ]:
%%writefile gen_track.py
import gc
import torch
import torch.multiprocessing as mp
import zipfile
import shutil
import numpy as np
import json
import zarr
from pathlib import Path
import os
from hoct import load_model, predict
from tracksdata.functional import TilingScheme

def load_masks(name):
    mask_path = "/kaggle/input/datasets/i2nfinit3y/biohub-focus3d-masks/" + f"{name}.npz" 
    z = np.load(mask_path)
    shape = tuple(int(v) for v in z["shape"])        # (64, 256, 256)
    
    T = 100
    full_mask = np.zeros((T, *shape), dtype=np.int16)
    
    # 2. 遍历所有时间帧恢复 mask
    for t in range(T):
        idx_key, lab_key = f"i{t}", f"l{t}"
        
        # 避免某些帧没有检测目标导致 key 不存在，做一层保护
        if idx_key in z and lab_key in z:
            idx, lab = z[idx_key], z[lab_key]
            
            # 将第 t 帧展平并赋值，0 为背景，1..K 为实例 ID
            full_mask[t].ravel()[idx] = lab

    return full_mask

def worker(gpu_id: int, task_queue: mp.Queue):
    # 限制每个进程使用的 CPU 线程数（2个进程 x 2线程 = 4线程，刚好占满 CPU）
    torch.cuda.set_device(gpu_id)
    torch.set_num_threads(2)
    
    device = f"cuda:{gpu_id}"
    model = load_model(device=device)

    tiling_scheme = TilingScheme(
            tile_shape=(5, 32, 128, 128),
            overlap_shape=(1, 8, 16, 16),
        )

    while True:
        sample_name = task_queue.get()
        if sample_name is None:  # 收到退出信号
            break

        full_mask = load_masks(sample_name)
        with torch.inference_mode():
            solution_graph = predict(
                model, 
                labels=full_mask, 
                scale=(1.0, 1.625, 0.40625, 0.40625), 
                max_delta_t=1,
                tiling_scheme=tiling_scheme
            )
        geff_path = f"{sample_name}.geff"
        zip_path = f"{sample_name}.zip"
        solution_graph.to_geff(geff_path, overwrite=True)

        # 2. 压缩成 .zip 文件
        with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            if os.path.isdir(geff_path):
                # 如果 geff 是目录，递归将内部所有文件打包
                for root, _, files in os.walk(geff_path):
                    for file in files:
                        abs_file = os.path.join(root, file)
                        # arcname 保持相对路径，解压时保持内部结构
                        rel_path = os.path.relpath(abs_file, start=geff_path)
                        zf.write(abs_file, arcname=os.path.join(os.path.basename(geff_path), rel_path))
            elif os.path.isfile(geff_path):
                # 如果 geff 是单文件
                zf.write(geff_path, arcname=os.path.basename(geff_path))

        # 3. 顺便删除原来的 geff 文件/文件夹
        if os.path.isdir(geff_path):
            shutil.rmtree(geff_path)
        elif os.path.isfile(geff_path):
            os.remove(geff_path)
        
        del full_mask, solution_graph
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()  # 清理跨进程共享内存（若存在多进程 DataLoader/切片）

        print(f"[{device}] sample {sample_name} done")

if __name__ == "__main__":
    # CUDA 在多进程中必须使用 'spawn' 启动模式
    mp.set_start_method("spawn", force=True)

    folder = "/kaggle/input/datasets/i2nfinit3y/biohub-focus3d-masks"
    files_list = [f.split(".")[0] for f in os.listdir(folder) if f.endswith(".npz")]
    num_gpus = 2

    # 创建任务队列并载入所有任务
    task_queue = mp.Queue()
    for sample_name in files_list:
        task_queue.put(sample_name)

    # 放入终止标志（Poison Pill），每个 worker 对应一个 None
    for _ in range(num_gpus):
        task_queue.put(None)

    # 启动 2 个 Worker 进程，分别绑定 cuda:0 和 cuda:1
    processes = []
    for gpu_id in range(num_gpus):
        p = mp.Process(target=worker, args=(gpu_id, task_queue))
        p.start()
        processes.append(p)

    # 等待所有进程处理完毕
    for p in processes:
        p.join()

In [ ]:
!python gen_track.py